In [6]:
import pandas as pd
import yfinance as yf
from pandas.tseries.offsets import BDay

start_date = '2019-06-11'
end_date   = '2025-08-10'  # current date in Europe/Paris timezone

# WTI price from DataHub (CSV) – example using FRED API if easier
wti = pd.read_csv(
    'https://raw.githubusercontent.com/datasets/oil-prices/master/data/wti-daily.csv',
    parse_dates=['Date']
).set_index('Date').loc[start_date:end_date] 
wti.rename(columns={'Price': 'WTI'}, inplace=True)

# Alternative: WTI via FRED using fredapi (requires an API key)
# from fredapi import Fred
# fred = Fred(api_key='YOUR_FRED_KEY')
# wti = fred.get_series('DCOILWTICO', observation_start=start_date, observation_end=end_date).to_frame(name='WTI')

# S&P 500, gold, US Dollar index, 10‑yr yield, DJU via yfinance
tickers = {
    'Gold': 'GC=F',          # COMEX gold futures
    'SP 500': '^GSPC',        # S&P 500
    'US DOLLAR INDEX': 'DX-Y.NYB', # U.S. Dollar Index
    'US 10YR BOND': '^TNX',   # 10‑year Treasury yield
    'DJU': '^DJU'            # Dow Jones Utility Average
}
yf_data = yf.download(
    list(tickers.values()),
    start=start_date,
    end=end_date,
    progress=False,
    auto_adjust=True
)['Close']
yf_data.rename(columns={v: k for k, v in tickers.items()}, inplace=True)

# Combine and align on business days
data = pd.concat([wti, yf_data], axis=1)
data = data.asfreq('B')  # business-day frequency
data = data.ffill()      # forward-fill weekends/holidays

# Inspect the updated dataset
print(data.tail())


              WTI  US DOLLAR INDEX         Gold          DJU       SP 500  \
Date                                                                        
2025-08-04  67.33        98.779999  3374.399902  1125.790039  6329.939941   
2025-08-05  67.33        98.779999  3381.899902  1110.060059  6299.189941   
2025-08-06  67.33        98.180000  3380.000000  1101.709961  6345.060059   
2025-08-07  67.33        98.400002  3400.300049  1118.189941  6340.000000   
2025-08-08  67.33        98.180000  3439.100098  1113.469971  6389.450195   

            US 10YR BOND  
Date                      
2025-08-04         4.200  
2025-08-05         4.196  
2025-08-06         4.220  
2025-08-07         4.244  
2025-08-08         4.285  


In [7]:
import pandas as pd

# --- 1) Load your original combined file (day-first) ---
orig = pd.read_csv('Data/COMBINED.csv', header=None)
# If you have headers, replace with: orig = pd.read_csv('Data/COMBINED.csv')
# Detect whether there is a header row; assuming first col is Date
if orig.columns[0] != 'Date':
    # No headers in the sample you showed; set them explicitly
    # Adjust the names to your exact columns if needed:
    orig.columns = ['Date','WTI','Gold','SP 500','US DOLLAR INDEX','US 10YR BOND','DJU']

# Force Date to datetime with dayfirst=True
orig['Date'] = pd.to_datetime(orig['Date'], dayfirst=True, errors='coerce')
orig = orig.set_index('Date').sort_index()

# --- 2) Build/clean your new data frame (from your fetch step) ---
# Assume you already built `data` like in your script:
# data = pd.concat([wti, yf_data], axis=1).asfreq('B').ffill()

# Make sure new data index is true datetime (no strings)
data.index = pd.to_datetime(data.index, errors='coerce')
data = data.sort_index()

# Optional: normalize ^TNX (Yahoo gives ~10×)
#if 'US 10YR BOND' in data.columns:
#    data['US 10YR BOND'] = data['US 10YR BOND'] / 10.0

# --- 3) Merge preferring the newest values for overlapping dates ---
# Option A: take latest non-null values on each date
merged = orig.combine_first(data)
# If you want new data to override old on overlap, do:
# merged = data.combine_first(orig)

merged = merged.sort_index()

# --- 4) Export with consistent DD-MM-YYYY format ---
# (a) If you want Date as first column (no index in the file):
out = merged.reset_index()
out['Date'] = out['Date'].dt.strftime('%d-%m-%Y')
out.to_csv('Data/COMBINED_updated.csv', index=False, float_format='%.6f')

# (b) If you prefer Date as index in the CSV:
# merged.to_csv('Data/COMBINED_updated.csv', date_format='%d-%m-%Y', float_format='%.6f')


C:\Users\tache_zfbqkx0\AppData\Local\Temp\ipykernel_28996\4096618756.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orig['Date'] = pd.to_datetime(orig['Date'], dayfirst=True, errors='coerce')


In [8]:
assert merged.index.dtype.kind == 'M'   # datetime index
print(out.head())                       # should show DD-MM-YYYY in the Date column


         Date     DJU   Gold   SP 500 US 10YR BOND US DOLLAR INDEX    WTI
0  04-01-2000   289.1  282.7  1411.75        6.499           100.1  25.55
1  05-01-2000  292.64  281.1   1413.5        6.599          100.05  24.91
2  06-01-2000   297.7  281.4     1404         6.54          100.34  24.78
3  07-01-2000     NaN  281.9   1460.5        6.513           100.5  24.22
4  10-01-2000  294.37  281.7     1475        6.554          100.65  24.67


In [5]:
import pandas as pd
import yfinance as yf

start_date = '2019-06-11'
end_date   = '2025-08-11'

# --- WTI from DataHub (slice to date range immediately) ---
wti = (
    pd.read_csv(
        'https://raw.githubusercontent.com/datasets/oil-prices/master/data/wti-daily.csv',
        parse_dates=['Date']
    )
    .rename(columns={'Date':'Date', 'Price':'WTI'})
    .set_index('Date')
    .sort_index()
    .loc[start_date:end_date]        # << limit here
)

# --- yfinance series (these already obey start/end) ---
tickers = {
    'SP500': '^GSPC',
    'Gold': 'GC=F',
    'DollarIndex': 'DX-Y.NYB',
    'Treasury10y': '^TNX',
    'DJU': '^DJU'
}
raw = yf.download(list(tickers.values()), start=start_date, end=end_date, progress=False, auto_adjust=True)
yf_data = raw['Close'] if 'Close' in raw.columns.get_level_values(0) else raw['Adj Close']
yf_data.rename(columns={v:k for k,v in tickers.items()}, inplace=True)

# Normalize ^TNX (Yahoo quotes ~10×)
if 'Treasury10y' in yf_data.columns:
    yf_data['Treasury10y'] = yf_data['Treasury10y'] / 10.0

# --- Combine, then align to business days in the target window only ---
data = pd.concat([wti, yf_data], axis=1).sort_index()
data = data.loc[start_date:end_date]            # << keep window tight BEFORE asfreq
data = data.asfreq('B').ffill()
# Drop any leading all-NaN rows just in case
data = data.loc[data.first_valid_index():]

# --- Load original combined (day-first) and standardize column names ---
orig = pd.read_csv('Data/COMBINED.csv', header=None)
if orig.columns[0] != 'Date':
    orig.columns = ['Date','WTI','Gold','SP 500','US Dollar Index','US 10YR BOND','DJU']
orig['Date'] = pd.to_datetime(orig['Date'], dayfirst=True, errors='coerce')
orig = orig.set_index('Date').sort_index()

# Standardize original names to match new names
name_map = {'SP 500': 'SP500', 'US Dollar Index': 'DollarIndex', 'US 10YR BOND': 'Treasury10y'}
orig = orig.rename(columns=name_map)

# --- Merge & enforce final start date ---
# Prefer the latest non-null values:
merged = orig.combine_first(data)               # or: data.combine_first(orig) to prefer new
merged = merged.sort_index()

# Keep only from 2019-06-11 onward
merged = merged.loc[start_date:end_date]

# If any old-name duplicates slipped in, drop them
for old, new in name_map.items():
    if old in merged.columns and new in merged.columns:
        merged.drop(columns=[old], inplace=True)

# --- Export with DD-MM-YYYY date format ---
out = merged.reset_index()
out['Date'] = out['Date'].dt.strftime('%d-%m-%Y')
out.to_csv('Data/COMBINED_updated.csv', index=False, float_format='%.6f')


C:\Users\tache_zfbqkx0\AppData\Local\Temp\ipykernel_28996\3950365893.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yf_data.rename(columns={v:k for k,v in tickers.items()}, inplace=True)
C:\Users\tache_zfbqkx0\AppData\Local\Temp\ipykernel_28996\3950365893.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yf_data['Treasury10y'] = yf_data['Treasury10y'] / 10.0
C:\Users\tache_zfbqkx0\AppData\Local\Temp\ipykernel_28996\3950365893.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure par

KeyError: 'Value based partial slicing on non-monotonic DatetimeIndexes with non-existing keys is not allowed.'